# Turkish News Summarization with Qwen3-0.6B and LoRA

This notebook prepares a balanced Turkish news summarization dataset from [`batubayk/TR-News`](https://huggingface.co/datasets/batubayk/TR-News), fine-tunes `Qwen/Qwen3-0.6B` with LoRA, and compares the adapted model with the base model.

Only the dataset's `content` and `abstract` fields are used. The notebook builds a 6,000-example training set and a fixed 120-example evaluation set. Turkish news text, reference summaries, and task instructions intentionally remain in Turkish; the notebook documentation and metric labels are in English.

By default, **Run All** downloads missing data, displays the exploratory and data-preparation outputs, rebuilds the fixed evaluation set, and scores the saved predictions. Model training and new prediction generation are disabled by default, so Run All will not train the model or launch a new generation run. The two switches are in the **Data and training settings** cell.


Repository note: saved outputs keep aggregate data-preparation and evaluation results. Raw article excerpts and individual dataset examples are omitted. Training and prediction generation were not rerun for this repository cleanup.

## 1. Download the dataset and keep the required columns


In [1]:
import csv
import gzip
import io
import os
from pathlib import Path
from urllib.request import urlopen

NOTEBOOK_MARKERS = (
    Path("notebooks") / "01-turkish-news-summarization.ipynb",
    Path("2LoraProje") / "notebooks" / "01-turkish-news-summarization.ipynb",
)
CANDIDATE_ROOTS = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
PROJECT_ROOT = next(
    (
        candidate
        for candidate in CANDIDATE_ROOTS
        if (candidate / "data" / "raw").is_dir()
        or (candidate / "data" / "processed").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    PROJECT_ROOT = next(
        (
            candidate
            for candidate in CANDIDATE_ROOTS
            if any((candidate / marker).is_file() for marker in NOTEBOOK_MARKERS)
        ),
        Path.cwd().resolve(),
    )

DATA_DIR = PROJECT_ROOT / "data" / "raw"
DATA_DIR.mkdir(parents=True, exist_ok=True)
DATASET_URL = "https://huggingface.co/datasets/batubayk/TR-News/resolve/main/{split}.csv"
SPLITS = ("train", "validation", "test")
SELECTED_COLUMNS = ("content", "abstract")
csv.field_size_limit(2**31 - 1)


def download_selected_columns(split, output_path):
    """Download one dataset split and retain only content and abstract."""
    if split not in SPLITS:
        raise ValueError(f"Unknown split {split!r}. Expected one of: {SPLITS}")

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if output_path.is_file():
        print(f"File already exists: {output_path.name} ({output_path.stat().st_size:,} bytes)")
        return output_path

    partial_path = output_path.with_name(output_path.name + ".partial")
    row_count = 0

    try:
        with urlopen(DATASET_URL.format(split=split), timeout=90) as response:
            source = io.TextIOWrapper(response, encoding="utf-8", newline="")
            reader = csv.reader(source)
            header = next(reader)
            missing_columns = [
                column for column in SELECTED_COLUMNS if column not in header
            ]
            if missing_columns:
                raise ValueError(
                    f"Missing expected columns {missing_columns}; received {header}."
                )
            column_indices = [header.index(column) for column in SELECTED_COLUMNS]

            with gzip.open(
                partial_path,
                "wt",
                encoding="utf-8",
                newline="",
                compresslevel=1,
            ) as destination:
                writer = csv.writer(destination)
                writer.writerow(SELECTED_COLUMNS)

                for row_count, row in enumerate(reader, start=1):
                    writer.writerow(
                        [
                            row[index] if index < len(row) else ""
                            for index in column_indices
                        ]
                    )
                    if row_count % 20_000 == 0:
                        print(f"{split}: processed {row_count:,} rows.", flush=True)

        os.replace(partial_path, output_path)
    except Exception:
        if partial_path.exists():
            partial_path.unlink()
        raise

    print(
        f"Download complete: {split} split, {row_count:,} rows, "
        f"{output_path.stat().st_size:,} bytes."
    )
    return output_path


dataset_paths = {
    split: DATA_DIR / f"TR-News_{split}_content_abstract.csv.gz"
    for split in SPLITS
}

for split, dataset_path in dataset_paths.items():
    download_selected_columns(split, dataset_path)

print("\nColumns kept: content and abstract")


File already exists: TR-News_train_content_abstract.csv.gz (264,031,872 bytes)
File already exists: TR-News_validation_content_abstract.csv.gz (14,032,788 bytes)
File already exists: TR-News_test_content_abstract.csv.gz (14,455,812 bytes)

Columns kept: content and abstract


## 2. Explore text lengths and summary ratios

Character counts and whitespace-separated word counts are descriptive approximations. They are not model-token counts; exact Qwen token lengths are measured during training-data preparation.

For each record, the word ratio is `(abstract words / content words) × 100`. Ratio bands include their lower bound and exclude their upper bound.


In [2]:
import bisect
import csv
import gzip
import math
import statistics

csv.field_size_limit(2**31 - 1)
fields = ("content", "abstract")
units = ("characters", "whitespace-separated words")
lengths = {
    split: {field: {unit: [] for unit in units} for field in fields}
    for split in SPLITS
}
pair_ratios = {split: {"characters": [], "words": []} for split in SPLITS}
row_counts = {}

for split, path in dataset_paths.items():
    count = 0
    with gzip.open(path, "rt", encoding="utf-8", newline="") as source:
        reader = csv.DictReader(source)
        assert reader.fieldnames == ["content", "abstract"], reader.fieldnames
        for row in reader:
            count += 1
            content = row["content"] or ""
            abstract = row["abstract"] or ""
            values = {"content": content, "abstract": abstract}
            for field, text in values.items():
                lengths[split][field]["characters"].append(len(text))
                lengths[split][field]["whitespace-separated words"].append(len(text.split()))
            content_chars = len(content)
            content_words = len(content.split())
            if content_chars:
                pair_ratios[split]["characters"].append(100 * len(abstract) / content_chars)
            if content_words:
                pair_ratios[split]["words"].append(100 * len(abstract.split()) / content_words)
    row_counts[split] = count


def describe(values):
    ordered = sorted(values)
    n = len(ordered)
    return {
        "n": n,
        "empty": sum(value == 0 for value in ordered),
        "mean": statistics.fmean(ordered) if n else 0,
        "p50": ordered[max(0, math.ceil(0.50 * n) - 1)] if n else 0,
        "p90": ordered[max(0, math.ceil(0.90 * n) - 1)] if n else 0,
        "p95": ordered[max(0, math.ceil(0.95 * n) - 1)] if n else 0,
        "p99": ordered[max(0, math.ceil(0.99 * n) - 1)] if n else 0,
        "max": ordered[-1] if n else 0,
    }


print(f"Total records: {sum(row_counts.values()):,}")
print("Rows by split:", ", ".join(f"{split}={row_counts[split]:,}" for split in SPLITS))
print("\nLENGTH SUMMARY (nearest-rank percentiles; word counts use whitespace splitting)")
print(f"{'Split':<12} {'Field':<10} {'Unit':<29} {'Empty':>8} {'Mean':>12} {'P50':>9} {'P90':>9} {'P95':>9} {'P99':>9} {'Max':>10}")
for split in SPLITS:
    for field in fields:
        for unit in units:
            stats = describe(lengths[split][field][unit])
            print(
                f"{split:<12} {field:<10} {unit:<29} {stats['empty']:>8,} "
                f"{stats['mean']:>12,.1f} {stats['p50']:>9,} {stats['p90']:>9,} "
                f"{stats['p95']:>9,} {stats['p99']:>9,} {stats['max']:>10,}"
            )

print("\nABSTRACT / CONTENT RATIO (matched records, percent)")
for split in SPLITS:
    char_stats = describe(pair_ratios[split]["characters"])
    word_stats = describe(pair_ratios[split]["words"])
    print(
        f"{split:<12} characters: median={char_stats['p50']:.1f}%, P90={char_stats['p90']:.1f}% | "
        f"words: median={word_stats['p50']:.1f}%, P90={word_stats['p90']:.1f}%"
    )


ratio_bounds = (5, 10, 15, 20, 25, 30, 40, 50, 75, 100)
ratio_labels = (
    "< 5%", "5-<10%", "10-<15%", "15-<20%", "20-<25%",
    "25-<30%", "30-<40%", "40-<50%", "50-<75%", "75-<100%", ">= 100%",
)
ratio_counts_by_split = {}
for split in SPLITS:
    bucket_counts = [0] * (len(ratio_bounds) + 1)
    for ratio in pair_ratios[split]["words"]:
        bucket_counts[bisect.bisect_right(ratio_bounds, ratio)] += 1
    ratio_counts_by_split[split] = bucket_counts

print("\nWORD-RATIO DISTRIBUTION (abstract words / content words)")
print(
    f"{'Ratio band':<13} {'Train count (share)':>21} {'Validation count (share)':>27} "
    f"{'Test count (share)':>21} {'All count (share)':>21}"
)
split_totals = {split: len(pair_ratios[split]["words"]) for split in SPLITS}
all_total = sum(split_totals.values())
for index, label in enumerate(ratio_labels):
    cells = []
    for split in SPLITS:
        count = ratio_counts_by_split[split][index]
        share = 100 * count / split_totals[split] if split_totals[split] else 0
        cells.append(f"{count:,} ({share:.1f}%)")
    all_count = sum(ratio_counts_by_split[split][index] for split in SPLITS)
    all_share = 100 * all_count / all_total if all_total else 0
    all_cell = f"{all_count:,} ({all_share:.1f}%)"
    print(
        f"{label:<13} {cells[0]:>21} {cells[1]:>27} "
        f"{cells[2]:>21} {all_cell:>21}"
    )

print("\nTRAIN CHARACTER LENGTH DISTRIBUTION (records per bucket)")
bounds = (256, 512, 1_024, 2_048, 4_096, 8_192, 16_384, 32_768, 65_536)
for field in fields:
    values = lengths["train"][field]["characters"]
    bins = [0] * (len(bounds) + 1)
    for value in values:
        bins[bisect.bisect_left(bounds, value)] += 1
    largest = max(bins) if bins else 1
    print(f"\n{field}:")
    for index, count in enumerate(bins):
        label = f"<= {bounds[index]:,}" if index < len(bounds) else f"> {bounds[-1]:,}"
        bar = "#" * max(1, round(30 * count / largest)) if count else ""
        percent = 100 * count / len(values) if values else 0
        print(f"  {label:>10}: {count:>8,} ({percent:5.1f}%) {bar}")

Total records: 307,562
Rows by split: train=277,573, validation=14,610, test=15,379

LENGTH SUMMARY (nearest-rank percentiles; word counts use whitespace splitting)
Split        Field      Unit                             Empty         Mean       P50       P90       P95       P99        Max
train        content    characters                           0      1,882.0     1,210     3,782     5,275    10,892    248,304
train        content    whitespace-separated words           0        240.9       154       485       679     1,407     30,735
train        abstract   characters                           0        165.6       150       261       311       435      1,325
train        abstract   whitespace-separated words           0         21.1        19        34        40        56        179
validation   content    characters                           0      1,900.1     1,225     3,891     5,361    10,545     84,799
validation   content    whitespace-separated words           0        243

### Dataset source

Baykara and Güngör (2022), [Abstractive text summarization and new large-scale datasets for agglutinative languages: Turkish and Hungarian](https://doi.org/10.1007/s10579-021-09568-y).


In [3]:
import bisect
import csv
import gzip
from pathlib import Path
from IPython.display import Markdown, display

project_root = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "raw").is_dir()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not locate the project folder containing data/raw.")
data_dir = project_root / "data" / "raw"
splits = ("train", "validation", "test")

ratio_bounds = (5, 10, 15, 20, 25, 30, 40, 50, 75, 100)
ratio_labels = (
    "< 5%",
    "5–<10%",
    "10–<15%",
    "15–<20%",
    "20–<25%",
    "25–<30%",
    "30–<40%",
    "40–<50%",
    "50–<75%",
    "75–<100%",
    ">= 100%",
)

counts_by_split = {}
valid_rows_by_split = {}
zero_content_word_rows = {}

csv.field_size_limit(2**31 - 1)

for split in splits:
    file_path = data_dir / f"TR-News_{split}_content_abstract.csv.gz"
    if not file_path.is_file():
        raise FileNotFoundError(
            f"Dataset file not found: {file_path}. Run the download cell first."
        )
    bucket_counts = [0] * len(ratio_labels)
    valid_rows = 0
    zero_content_words = 0

    with gzip.open(file_path, "rt", encoding="utf-8", newline="") as source:
        reader = csv.DictReader(source)

        if reader.fieldnames != ["content", "abstract"]:
            raise ValueError(
                f"Unexpected columns in {file_path}: {reader.fieldnames}"
            )

        for row in reader:
            content_words = len((row["content"] or "").split())
            abstract_words = len((row["abstract"] or "").split())

            if content_words == 0:
                zero_content_words += 1
                continue

            ratio_percent = 100 * abstract_words / content_words
            bucket_index = bisect.bisect_right(ratio_bounds, ratio_percent)
            bucket_counts[bucket_index] += 1
            valid_rows += 1

    counts_by_split[split] = bucket_counts
    valid_rows_by_split[split] = valid_rows
    zero_content_word_rows[split] = zero_content_words

all_counts = [
    sum(counts_by_split[split][index] for split in splits)
    for index in range(len(ratio_labels))
]
all_valid_rows = sum(valid_rows_by_split.values())

table_lines = [
    "| Abstract / content word ratio | Train | Validation | Test | All |",
    "|---|---:|---:|---:|---:|",
]

for index, label in enumerate(ratio_labels):
    cells = []

    for split in splits:
        count = counts_by_split[split][index]
        total = valid_rows_by_split[split]
        share = 100 * count / total if total else 0
        cells.append(f"{count:,} ({share:.1f}%)")

    all_count = all_counts[index]
    all_share = 100 * all_count / all_valid_rows if all_valid_rows else 0
    all_cell = f"{all_count:,} ({all_share:.1f}%)"

    table_lines.append(
        f"| {label} | {cells[0]} | {cells[1]} | {cells[2]} | {all_cell} |"
    )

display(Markdown("\n".join(table_lines)))

print(
    "Rows excluded because content has zero words:",
    ", ".join(f"{split}={zero_content_word_rows[split]:,}" for split in splits),
)


| Abstract / content word ratio | Train | Validation | Test | All |
|---|---:|---:|---:|---:|
| < 5% | 38,471 (13.9%) | 2,076 (14.2%) | 2,083 (13.5%) | 42,630 (13.9%) |
| 5–<10% | 68,123 (24.5%) | 3,571 (24.4%) | 3,791 (24.7%) | 75,485 (24.5%) |
| 10–<15% | 58,040 (20.9%) | 3,042 (20.8%) | 3,280 (21.3%) | 64,362 (20.9%) |
| 15–<20% | 41,007 (14.8%) | 2,172 (14.9%) | 2,283 (14.8%) | 45,462 (14.8%) |
| 20–<25% | 27,407 (9.9%) | 1,423 (9.7%) | 1,499 (9.7%) | 30,329 (9.9%) |
| 25–<30% | 16,918 (6.1%) | 927 (6.3%) | 937 (6.1%) | 18,782 (6.1%) |
| 30–<40% | 15,859 (5.7%) | 788 (5.4%) | 857 (5.6%) | 17,504 (5.7%) |
| 40–<50% | 5,840 (2.1%) | 315 (2.2%) | 300 (2.0%) | 6,455 (2.1%) |
| 50–<75% | 4,012 (1.4%) | 210 (1.4%) | 232 (1.5%) | 4,454 (1.4%) |
| 75–<100% | 787 (0.3%) | 38 (0.3%) | 50 (0.3%) | 875 (0.3%) |
| >= 100% | 1,109 (0.4%) | 48 (0.3%) | 67 (0.4%) | 1,224 (0.4%) |

Rows excluded because content has zero words: train=0, validation=0, test=0


In [ ]:
import bisect
import csv
import gzip
import random
from html import escape
from pathlib import Path
from IPython.display import HTML, display

split = "train"
project_root = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "raw").is_dir()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not locate the project folder containing data/raw.")
data_dir = project_root / "data" / "raw"
file_path = data_dir / f"TR-News_{split}_content_abstract.csv.gz"
if not file_path.is_file():
    raise FileNotFoundError(f"Dataset file not found: {file_path}. Run the download cell first.")

ratio_bounds = (5, 10, 15, 20, 25, 30, 40, 50, 75, 100)
ratio_labels = (
    "< 5%",
    "5–<10%",
    "10–<15%",
    "15–<20%",
    "20–<25%",
    "25–<30%",
    "30–<40%",
    "40–<50%",
    "50–<75%",
    "75–<100%",
    ">= 100%",
)

csv.field_size_limit(2**31 - 1)

rng = random.Random(42)
seen_per_bucket = [0] * len(ratio_labels)
examples = [None] * len(ratio_labels)

with gzip.open(file_path, "rt", encoding="utf-8", newline="") as source:
    reader = csv.DictReader(source)

    if reader.fieldnames != ["content", "abstract"]:
        raise ValueError(f"Unexpected columns: {reader.fieldnames}")

    for row in reader:
        content = row["content"] or ""
        abstract = row["abstract"] or ""

        content_words = len(content.split())
        abstract_words = len(abstract.split())

        if content_words == 0:
            continue

        ratio_percent = 100 * abstract_words / content_words
        bucket_index = bisect.bisect_right(ratio_bounds, ratio_percent)

        # Reservoir sampling selects one random example from each bucket.
        seen_per_bucket[bucket_index] += 1
        if rng.randrange(seen_per_bucket[bucket_index]) == 0:
            examples[bucket_index] = (
                content,
                abstract,
                ratio_percent,
                content_words,
                abstract_words,
            )

html_parts = [f"<h3>Random examples by word-ratio band ({escape(split)} split)</h3>"]

for index, label in enumerate(ratio_labels):
    sample = examples[index]

    if sample is None:
        html_parts.append(f"<p><b>{escape(label)}:</b> No records in this band.</p>")
        continue

    content, abstract, ratio, content_words, abstract_words = sample
    content_preview = content[:1000]
    truncation_note = (
        " [content excerpt truncated at 1,000 characters]"
        if len(content) > 1000
        else ""
    )

    html_parts.append(
        f"""
        <section style="margin: 1em 0; padding: 0.8em; border: 1px solid #ccc;">
          <h4>{escape(label)} — {seen_per_bucket[index]:,} records</h4>
          <p>
            Ratio: {ratio:.2f}% |
            Content words: {content_words:,} |
            Abstract words: {abstract_words:,}
          </p>
          <p><b>Content{truncation_note}</b></p>
          <pre style="white-space: pre-wrap;">{escape(content_preview)}</pre>
          <p><b>Abstract</b></p>
          <pre style="white-space: pre-wrap;">{escape(abstract)}</pre>
        </section>
        """
    )

display(HTML("\n".join(html_parts)))

In [5]:
import bisect
import csv
import gzip
from pathlib import Path
from IPython.display import Markdown, display

split = "train"
project_root = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "raw").is_dir()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not locate the project folder containing data/raw.")
data_dir = project_root / "data" / "raw"

file_path = data_dir / f"TR-News_{split}_content_abstract.csv.gz"
if not file_path.is_file():
    raise FileNotFoundError(f"Dataset file not found: {file_path}. Run the download cell first.")

# Content-length groups are based on whitespace-separated word counts.
content_upper_bounds = (100, 200, 400, 800)
content_labels = (
    "Under 100",
    "100-199",
    "200-399",
    "400-799",
    "800 or more",
)

# Ratio = abstract word count / content word count * 100.
ratio_bounds = (5, 10, 15, 20, 25, 30, 40, 50, 75, 100)
ratio_labels = (
    "Under 5%",
    "5 to under 10%",
    "10 to under 15%",
    "15 to under 20%",
    "20 to under 25%",
    "25 to under 30%",
    "30 to under 40%",
    "40 to under 50%",
    "50 to under 75%",
    "75 to under 100%",
    "100% or more",
)

counts = [
    [0 for _ in ratio_labels]
    for _ in content_labels
]
content_group_totals = [0 for _ in content_labels]
zero_content_word_rows = 0

csv.field_size_limit(2**31 - 1)

with gzip.open(file_path, "rt", encoding="utf-8", newline="") as source:
    reader = csv.DictReader(source)

    if reader.fieldnames != ["content", "abstract"]:
        raise ValueError(f"Unexpected columns: {reader.fieldnames}")

    for row in reader:
        content_words = len((row["content"] or "").split())
        abstract_words = len((row["abstract"] or "").split())

        if content_words == 0:
            zero_content_word_rows += 1
            continue

        content_group = bisect.bisect_right(
            content_upper_bounds,
            content_words,
        )
        ratio_percent = 100 * abstract_words / content_words
        ratio_group = bisect.bisect_right(ratio_bounds, ratio_percent)

        counts[content_group][ratio_group] += 1
        content_group_totals[content_group] += 1

ratio_totals = [
    sum(counts[row][column] for row in range(len(content_labels)))
    for column in range(len(ratio_labels))
]
total_records = sum(content_group_totals)

table_lines = [
    f"### Content length and summary ratio distribution — {split}",
    "",
    "Each cell shows **record count (percentage within that content-length group)**.",
    "",
    "| Content words | Records | " + " | ".join(ratio_labels) + " |",
    "|---|---:|" + "---:|" * len(ratio_labels),
]

for row, content_label in enumerate(content_labels):
    row_total = content_group_totals[row]
    cells = []

    for column in range(len(ratio_labels)):
        count = counts[row][column]
        share = 100 * count / row_total if row_total else 0
        cells.append(f"{count:,} ({share:.1f}%)")

    table_lines.append(
        f"| {content_label} | {row_total:,} | " + " | ".join(cells) + " |"
    )

overall_cells = []
for count in ratio_totals:
    share = 100 * count / total_records if total_records else 0
    overall_cells.append(f"{count:,} ({share:.1f}%)")

table_lines.append(
    f"| **All lengths** | **{total_records:,}** | "
    + " | ".join(f"**{cell}**" for cell in overall_cells)
    + " |"
)

display(Markdown("\n".join(table_lines)))
print(f"Rows skipped because content has zero words: {zero_content_word_rows:,}")

### Content length and summary ratio distribution — train

Each cell shows **record count (percentage within that content-length group)**.

| Content words | Records | Under 5% | 5 to under 10% | 10 to under 15% | 15 to under 20% | 20 to under 25% | 25 to under 30% | 30 to under 40% | 40 to under 50% | 50 to under 75% | 75 to under 100% | 100% or more |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| Under 100 | 81,304 | 109 (0.1%) | 1,636 (2.0%) | 9,059 (11.1%) | 16,153 (19.9%) | 16,530 (20.3%) | 12,744 (15.7%) | 13,735 (16.9%) | 5,505 (6.8%) | 3,940 (4.8%) | 785 (1.0%) | 1,108 (1.4%) |
| 100-199 | 89,917 | 1,392 (1.5%) | 18,832 (20.9%) | 32,612 (36.3%) | 20,903 (23.2%) | 9,802 (10.9%) | 3,947 (4.4%) | 2,029 (2.3%) | 327 (0.4%) | 70 (0.1%) | 2 (0.0%) | 1 (0.0%) |
| 200-399 | 66,470 | 10,332 (15.5%) | 35,971 (54.1%) | 14,985 (22.5%) | 3,810 (5.7%) | 1,044 (1.6%) | 224 (0.3%) | 94 (0.1%) | 8 (0.0%) | 2 (0.0%) | 0 (0.0%) | 0 (0.0%) |
| 400-799 | 30,088 | 17,519 (58.2%) | 11,045 (36.7%) | 1,349 (4.5%) | 140 (0.5%) | 31 (0.1%) | 3 (0.0%) | 1 (0.0%) | 0 (0.0%) | 0 (0.0%) | 0 (0.0%) | 0 (0.0%) |
| 800 or more | 9,794 | 9,119 (93.1%) | 639 (6.5%) | 35 (0.4%) | 1 (0.0%) | 0 (0.0%) | 0 (0.0%) | 0 (0.0%) | 0 (0.0%) | 0 (0.0%) | 0 (0.0%) | 0 (0.0%) |
| **All lengths** | **277,573** | **38,471 (13.9%)** | **68,123 (24.5%)** | **58,040 (20.9%)** | **41,007 (14.8%)** | **27,407 (9.9%)** | **16,918 (6.1%)** | **15,859 (5.7%)** | **5,840 (2.1%)** | **4,012 (1.4%)** | **787 (0.3%)** | **1,109 (0.4%)** |

Rows skipped because content has zero words: 0


In [ ]:
import csv
import gzip
import random
from html import escape
from pathlib import Path
from IPython.display import HTML, display

split = "train"
target_lengths = (100, 200, 300, 400, 500, 600)
samples_per_length = 3
project_root = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "raw").is_dir()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not locate the project folder containing data/raw.")
data_dir = project_root / "data" / "raw"
file_path = data_dir / f"TR-News_{split}_content_abstract.csv.gz"
if not file_path.is_file():
    raise FileNotFoundError(f"Dataset file not found: {file_path}. Run the download cell first.")

csv.field_size_limit(2**31 - 1)

rng = random.Random(42)
samples = {length: [] for length in target_lengths}
matches_seen = {length: 0 for length in target_lengths}

with gzip.open(file_path, "rt", encoding="utf-8", newline="") as source:
    reader = csv.DictReader(source)

    if reader.fieldnames != ["content", "abstract"]:
        raise ValueError(f"Unexpected columns: {reader.fieldnames}")

    for row in reader:
        content = row["content"] or ""
        abstract = row["abstract"] or ""
        content_words = len(content.split())

        if content_words not in samples:
            continue

        matches_seen[content_words] += 1
        current_samples = samples[content_words]

        if len(current_samples) < samples_per_length:
            current_samples.append((content, abstract))
        else:
            replacement_index = rng.randrange(matches_seen[content_words])
            if replacement_index < samples_per_length:
                current_samples[replacement_index] = (content, abstract)

html_parts = [
    f"<h3>Examples with exact content word counts ({escape(split)} split)</h3>"
]

for target_length in target_lengths:
    selected = samples[target_length]
    html_parts.append(
        f"<h4>{target_length} words — "
        f"{matches_seen[target_length]:,} matching records</h4>"
    )

    if not selected:
        html_parts.append("<p>No matching records found.</p>")
        continue

    for example_number, (content, abstract) in enumerate(selected, start=1):
        abstract_words = len(abstract.split())
        ratio = 100 * abstract_words / target_length

        html_parts.append(
            f"""
            <details style="margin: 0.5em 0;">
              <summary>Example {example_number}</summary>
              <p>Abstract/content word ratio: {ratio:.1f}%</p>
              <p><b>Content</b></p>
              <pre style="white-space: pre-wrap;">{escape(content)}</pre>
              <p><b>Abstract</b></p>
              <pre style="white-space: pre-wrap;">{escape(abstract)}</pre>
            </details>
            """
        )

display(HTML("\n".join(html_parts)))

## 3. Prepare the balanced training set

The preparation cell selects 6,000 records from the training split. It balances the three summary-length classes across twelve 25-word content-length bins, keeps at most 390 combined content-and-abstract words, and verifies the Qwen token limit before sampling.

The Turkish instruction wording is part of the training data: each class uses its canonical phrase for 70% of examples and three alternatives for the remaining 30%. Instructions appear before the article in 80% of examples and after it in 20%.


In [7]:
import csv
import gzip
import random
from collections import Counter, defaultdict
from pathlib import Path

from transformers import AutoTokenizer

RANDOM_SEED = 42
TARGET_TOTAL_RECORDS = 6_000
TARGET_CLASS_COUNTS = {"short": 1_200, "medium": 3_600, "long": 1_200}
SUMMARY_CLASSES = ("short", "medium", "long")
MIN_CONTENT_WORDS = 1
MAX_CONTENT_WORDS = 300
MAX_TOTAL_WORDS = 390
CONTENT_BIN_WIDTH = 25
CONTENT_BIN_COUNT = MAX_CONTENT_WORDS // CONTENT_BIN_WIDTH
CONTENT_BIN_LABELS = tuple(
    f"{index * CONTENT_BIN_WIDTH + 1}-{(index + 1) * CONTENT_BIN_WIDTH}"
    for index in range(CONTENT_BIN_COUNT)
)
SHORT_SUMMARY_MAX_WORDS = 12
MEDIUM_SUMMARY_MAX_WORDS = 25
MAX_SEQUENCE_LENGTH = 1_024
TOKEN_LENGTH_SAFETY_MARGIN = 8
TOKENIZER_BATCH_SIZE = 256
TOKENIZER_MODEL_NAME = "Qwen/Qwen3-0.6B"

PROMPT_VARIANTS = {
    "short": [
        ("K\u0131sa \u00f6zetle.", 70),
        ("K\u0131saca \u00f6zetle.", 10),
        ("K\u0131sa bir \u00f6zet haz\u0131rla.", 10),
        ("\u00d6nemli noktalar\u0131 k\u0131sa bi\u00e7imde aktar.", 10),
    ],
    "medium": [
        ("\u00d6zetle.", 70),
        ("Metnin ana fikrini ve temel noktalar\u0131n\u0131 \u00f6zetle.", 10),
        ("Dengeli uzunlukta bir \u00f6zet \u00e7\u0131kar.", 10),
        ("Gerekli ayr\u0131nt\u0131lar\u0131 koruyarak \u00f6zetle.", 10),
    ],
    "long": [
        ("Uzun \u00f6zetle.", 70),
        ("Ayr\u0131nt\u0131lar\u0131 koruyarak kapsaml\u0131 bi\u00e7imde \u00f6zetle.", 10),
        ("\u00d6nemli noktalar\u0131 atlamadan daha uzun bir \u00f6zet haz\u0131rla.", 10),
        ("Ana fikri ve destekleyici ayr\u0131nt\u0131lar\u0131 i\u00e7eren geni\u015f bir \u00f6zet yaz.", 10),
    ],
}
INSTRUCTION_POSITION_WEIGHTS = {"prefix": 80, "suffix": 20}

project_root = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "raw").is_dir()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not locate the project folder containing data/raw.")
source_path = project_root / "data" / "raw" / "TR-News_train_content_abstract.csv.gz"
output_path = project_root / "data" / "processed" / "TR-News_6000_summary_length_content_balanced.csv.gz"
csv.field_size_limit(2**31 - 1)


def assign_summary_class(abstract_words):
    """Assign a class using only the absolute abstract word count."""
    if abstract_words <= SHORT_SUMMARY_MAX_WORDS:
        return "short"
    if abstract_words <= MEDIUM_SUMMARY_MAX_WORDS:
        return "medium"
    return "long"


def parse_candidate(row):
    """Return a word-valid candidate and its summary/content-length bucket."""
    content = row["content"] or ""
    abstract = row["abstract"] or ""
    content_words = len(content.split())
    abstract_words = len(abstract.split())
    if not MIN_CONTENT_WORDS <= content_words <= MAX_CONTENT_WORDS:
        return None
    if abstract_words == 0 or content_words + abstract_words > MAX_TOTAL_WORDS:
        return None

    bin_index = (content_words - 1) // CONTENT_BIN_WIDTH
    summary_class = assign_summary_class(abstract_words)
    return (summary_class, bin_index), {
        "summary_class": summary_class,
        "content": content,
        "abstract": abstract,
        "content_words": content_words,
        "abstract_words": abstract_words,
        "total_words": content_words + abstract_words,
        "content_length_bin": CONTENT_BIN_LABELS[bin_index],
        "content_bin_index": bin_index,
    }


data_tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_MODEL_NAME,
    use_fast=True,
    local_files_only=False,
)
IM_END_ID = data_tokenizer.convert_tokens_to_ids("<|im_end|>")
if IM_END_ID is None or IM_END_ID == data_tokenizer.unk_token_id:
    raise ValueError("The tokenizer does not define the Qwen <|im_end|> token.")


def token_id_rows(token_output):
    """Normalize tokenization output into a list of token-ID lists."""
    if hasattr(token_output, "get"):
        token_output = token_output.get("input_ids", token_output)
    if hasattr(token_output, "tolist"):
        token_output = token_output.tolist()
    if token_output and isinstance(token_output[0], int):
        token_output = [token_output]
    return [[int(token_id) for token_id in row] for row in token_output]


longest_prompt_by_class = {}
for summary_class, variants in PROMPT_VARIANTS.items():
    measured = []
    for phrase, _ in variants:
        ids = data_tokenizer.apply_chat_template(
            [{"role": "user", "content": phrase}],
            tokenize=True,
            add_generation_prompt=True,
            enable_thinking=False,
            return_dict=False,
        )
        measured.append((len(token_id_rows(ids)[0]), phrase))
    longest_prompt_by_class[summary_class] = max(measured)[1]


def token_valid_candidate_batch(candidate_batch, rejected_by_bucket=None):
    """Check the longer of the prefix/suffix instruction prompt token lengths."""
    conversations = []
    for _, record in candidate_batch:
        phrase = longest_prompt_by_class[record["summary_class"]]
        conversations.append([{"role": "user", "content": f"{phrase}\n\n{record['content']}"}])
        conversations.append([{"role": "user", "content": f"{record['content']}\n\n{phrase}"}])

    prompt_ids = token_id_rows(
        data_tokenizer.apply_chat_template(
            conversations,
            tokenize=True,
            add_generation_prompt=True,
            enable_thinking=False,
            return_dict=False,
        )
    )
    response_ids = token_id_rows(
        data_tokenizer(
            [record["abstract"] for _, record in candidate_batch],
            add_special_tokens=False,
            padding=False,
        )
    )
    if len(prompt_ids) != 2 * len(candidate_batch) or len(response_ids) != len(candidate_batch):
        raise RuntimeError("Tokenizer returned an unexpected batch size.")

    valid = []
    for index, ((bucket_key, record), target_ids) in enumerate(zip(candidate_batch, response_ids)):
        total_tokens = max(len(prompt_ids[2 * index]), len(prompt_ids[2 * index + 1])) + len(target_ids) + 1
        upper_bound = total_tokens + TOKEN_LENGTH_SAFETY_MARGIN
        if upper_bound > MAX_SEQUENCE_LENGTH:
            if rejected_by_bucket is not None:
                rejected_by_bucket[bucket_key] += 1
            continue
        record["token_count_upper_bound"] = upper_bound
        valid.append((bucket_key, record))
    return valid


def iter_token_valid_candidates(reader, rejected_by_bucket=None, pass_label="Token scan"):
    """Yield token-valid records in batches without loading all text into memory."""
    batch = []
    row_count = 0
    for row_count, row in enumerate(reader, start=1):
        if row_count % 100_000 == 0:
            print(f"{pass_label}: scanned {row_count:,} source rows.", flush=True)
        candidate = parse_candidate(row)
        if candidate is None:
            continue
        batch.append(candidate)
        if len(batch) == TOKENIZER_BATCH_SIZE:
            yield from token_valid_candidate_batch(batch, rejected_by_bucket)
            batch = []
    if batch:
        yield from token_valid_candidate_batch(batch, rejected_by_bucket)
    print(f"{pass_label}: finished scanning {row_count:,} source rows.", flush=True)


def allocate_even_quotas(available_by_bin, target, rng):
    """Spread a class target evenly across bins, redistributing shortages."""
    quotas = Counter()
    remaining = target
    while remaining > 0:
        active = [i for i, count in available_by_bin.items() if quotas[i] < count]
        if not active:
            raise ValueError("Not enough examples to fill a summary class.")
        share, remainder = divmod(remaining, len(active))
        if share == 0:
            rng.shuffle(active)
            for i in active[:remainder]:
                quotas[i] += 1
            remaining = 0
            continue
        saturated = [i for i in active if available_by_bin[i] - quotas[i] <= share]
        if saturated:
            for i in saturated:
                amount = available_by_bin[i] - quotas[i]
                quotas[i] += amount
                remaining -= amount
            continue
        for i in active:
            quotas[i] += share
        remaining -= share * len(active)
        if remainder:
            rng.shuffle(active)
            for i in active[:remainder]:
                quotas[i] += 1
            remaining = 0
    return quotas


def build_weighted_variant_list(variants, record_count, rng):
    """Create an exact weighted list of instruction variants."""
    counts = [record_count * weight // 100 for _, weight in variants]
    remainders = [record_count * weight % 100 for _, weight in variants]
    remaining = record_count - sum(counts)
    order = sorted(range(len(variants)), key=lambda i: (-remainders[i], i))
    for i in order[:remaining]:
        counts[i] += 1
    phrases = []
    for (phrase, _), count in zip(variants, counts):
        phrases.extend([phrase] * count)
    rng.shuffle(phrases)
    return phrases


def build_instruction_positions(record_count, rng):
    """Create an exact 80/20 prefix/suffix instruction-position list."""
    prefix_count = round(record_count * INSTRUCTION_POSITION_WEIGHTS["prefix"] / 100)
    positions = ["prefix"] * prefix_count + ["suffix"] * (record_count - prefix_count)
    rng.shuffle(positions)
    return positions


# First pass: count candidates that satisfy word and token limits.
available_by_bucket = Counter()
token_limit_rejections = Counter()
with gzip.open(source_path, "rt", encoding="utf-8", newline="") as source_file:
    reader = csv.DictReader(source_file)
    if reader.fieldnames != ["content", "abstract"]:
        raise ValueError(f"Unexpected columns in {source_path}: {reader.fieldnames}")
    for bucket_key, _ in iter_token_valid_candidates(reader, token_limit_rejections, "Counting"):
        available_by_bucket[bucket_key] += 1

available_by_class = Counter()
for (summary_class, _), count in available_by_bucket.items():
    available_by_class[summary_class] += count
for summary_class in SUMMARY_CLASSES:
    required = TARGET_CLASS_COUNTS[summary_class]
    if available_by_class[summary_class] < required:
        raise ValueError(
            f"Not enough token-valid {summary_class} examples: "
            f"{available_by_class[summary_class]:,} available, {required:,} required."
        )

# Allocate each class evenly over all twelve 25-word content bins.
quota_rng = random.Random(RANDOM_SEED + 1)
quotas = Counter()
for summary_class in SUMMARY_CLASSES:
    availability = {
        i: available_by_bucket[(summary_class, i)]
        for i in range(CONTENT_BIN_COUNT)
    }
    class_quotas = allocate_even_quotas(
        availability,
        TARGET_CLASS_COUNTS[summary_class],
        quota_rng,
    )
    for bin_index, quota in class_quotas.items():
        if quota:
            quotas[(summary_class, bin_index)] = quota

# Second pass: reservoir-sample the exact quota from every class/bin cell.
reservoirs = {key: [] for key in quotas}
seen_by_bucket = Counter()
sample_rng = random.Random(RANDOM_SEED + 2)
with gzip.open(source_path, "rt", encoding="utf-8", newline="") as source_file:
    for bucket_key, record in iter_token_valid_candidates(
        csv.DictReader(source_file),
        pass_label="Sampling",
    ):
        if bucket_key not in reservoirs:
            continue
        seen_by_bucket[bucket_key] += 1
        samples = reservoirs[bucket_key]
        quota = quotas[bucket_key]
        if len(samples) < quota:
            samples.append(record)
        else:
            replacement = sample_rng.randrange(seen_by_bucket[bucket_key])
            if replacement < quota:
                samples[replacement] = record

for key, quota in quotas.items():
    if len(reservoirs[key]) != quota:
        raise RuntimeError(f"Sampling did not fill bucket: {key}")
selected_records = [record for bucket in reservoirs.values() for record in bucket]
if len(selected_records) != TARGET_TOTAL_RECORDS:
    raise RuntimeError(f"Expected 6,000 records, found {len(selected_records):,}.")

# Assign 70% canonical wording, 30% alternatives, and 80/20 instruction position.
selected_by_class = defaultdict(list)
for record in selected_records:
    selected_by_class[record["summary_class"]].append(record)
variant_rng = random.Random(RANDOM_SEED + 3)
for summary_class in SUMMARY_CLASSES:
    records = selected_by_class[summary_class]
    instructions = build_weighted_variant_list(
        PROMPT_VARIANTS[summary_class], len(records), variant_rng
    )
    positions = build_instruction_positions(len(records), variant_rng)
    variant_rng.shuffle(records)
    for record, instruction, position in zip(records, instructions, positions):
        record["instruction"] = instruction
        record["instruction_position"] = position

sample_rng.shuffle(selected_records)
selected_by_bin_class = Counter(
    (record["content_length_bin"], record["summary_class"])
    for record in selected_records
)
selected_by_instruction = Counter(
    (record["summary_class"], record["instruction"])
    for record in selected_records
)
selected_by_position = Counter(
    (record["summary_class"], record["instruction_position"])
    for record in selected_records
)

for record in selected_records:
    record.pop("content_bin_index", None)

output_path.parent.mkdir(parents=True, exist_ok=True)
fieldnames = [
    "summary_class", "instruction", "instruction_position", "content", "abstract",
    "content_words", "abstract_words", "total_words", "content_length_bin",
    "token_count_upper_bound",
]
with gzip.open(output_path, "wt", encoding="utf-8", newline="", compresslevel=1) as destination:
    writer = csv.DictWriter(destination, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(selected_records)

print(f"Saved {len(selected_records):,} records to {output_path.name}")
print("Word limits: 1-300 content words; no more than 390 combined words.")
print("Summary class counts (abstract length only):")
for summary_class in SUMMARY_CLASSES:
    count = sum(record["summary_class"] == summary_class for record in selected_records)
    print(f"  {summary_class}: {count:,} ({100 * count / len(selected_records):.1f}%)")
print(f"Token-valid candidates before sampling: {sum(available_by_bucket.values()):,}")
print(f"Rejected by the 1,024-token limit: {sum(token_limit_rejections.values()):,}")
print("Maximum selected token-count upper bound:", max(r["token_count_upper_bound"] for r in selected_records))

print("\nSelected counts by content bin and summary class:")
print(f"{'Content words':<14}{'Short':>10}{'Medium':>10}{'Long':>10}{'Total':>10}")
for bin_label in CONTENT_BIN_LABELS:
    counts = [selected_by_bin_class[(bin_label, label)] for label in SUMMARY_CLASSES]
    print(f"{bin_label:<14}{counts[0]:>10,}{counts[1]:>10,}{counts[2]:>10,}{sum(counts):>10,}")

print("\nInstruction wording counts:")
for summary_class in SUMMARY_CLASSES:
    print(f"{summary_class}:")
    for phrase, _ in PROMPT_VARIANTS[summary_class]:
        print(f"  {phrase}: {selected_by_instruction[(summary_class, phrase)]:,}")
print("\nInstruction positions:")
for summary_class in SUMMARY_CLASSES:
    print(
        f"{summary_class}: prefix={selected_by_position[(summary_class, 'prefix')]:,}, "
        f"suffix={selected_by_position[(summary_class, 'suffix')]:,}"
    )

print("\nPrepared training examples:")
for summary_class in SUMMARY_CLASSES:
    example = next(
        record
        for record in selected_records
        if record["summary_class"] == summary_class
    )
    content_excerpt = " ".join(example["content"].split())
    if len(content_excerpt) > 350:
        content_excerpt = content_excerpt[:347].rstrip() + "..."
    print(f"\n--- {summary_class.title()} summary ---")
    print("Instruction position:", example["instruction_position"])
    print("Instruction:", example["instruction"])
    print("Article excerpt:", content_excerpt)
    print("Reference summary:", example["abstract"])


Counting: scanned 100,000 source rows.
Counting: scanned 200,000 source rows.
Counting: finished scanning 277,573 source rows.
Sampling: scanned 100,000 source rows.
Sampling: scanned 200,000 source rows.
Sampling: finished scanning 277,573 source rows.
Saved 6,000 records to TR-News_6000_summary_length_content_balanced.csv.gz
Word limits: 1-300 content words; no more than 390 combined words.
Summary class counts (abstract length only):
  short: 1,200 (20.0%)
  medium: 3,600 (60.0%)
  long: 1,200 (20.0%)
Token-valid candidates before sampling: 214,312
Rejected by the 1,024-token limit: 138
Maximum selected token-count upper bound: 1015

Selected counts by content bin and summary class:
Content words      Short    Medium      Long     Total
1-25                 100       300       100       500
26-50                100       300       100       500
51-75                100       300       100       500
76-100               100       300       100       500
101-125              100      

## 4. Fine-tune Qwen3-0.6B with LoRA

The 6,000-example training set contains 1,200 short, 3,600 medium, and 1,200 long summaries. Classes are assigned by abstract word count: short is at most 12 words, medium is 13–25 words, and long is more than 25 words. Each of the twelve 25-word content-length bins targets the same class mix: 100 short, 300 medium, and 100 long examples.

Each class uses its canonical Turkish instruction for 70% of examples and alternative phrasings for 30%. The instruction appears before the article in 80% of examples and after it in 20%. The data-preparation cell checks word and Qwen-token limits before sampling.

Training is disabled by default. The default **Run All** workflow skips tokenization for training, model/Trainer setup, checkpoint resume, and `trainer.train()`. Set `RUN_TRAINING = True` in the settings cell only when you intentionally want to train or resume.


### Dependencies

The [Qwen3-0.6B model card](https://huggingface.co/Qwen/Qwen3-0.6B) requires Transformers 4.51.0 or later. The training workflow also uses PyTorch, PEFT, Datasets, and Accelerate. ROUGE and BERTScore evaluation dependencies are installed in the evaluation section.


### Data and training settings

Run All keeps `RUN_TRAINING = False` and `RUN_PREDICTION_GENERATION = False`, so it does not train the model or generate new predictions. It does run data analysis and preparation, rebuild the fixed evaluation set, and use saved predictions for scoring when available.

For an intentional training run, set `RUN_TRAINING = True` in this cell, then run the data/settings, tokenization, Trainer setup, and resume cells in order. The resume cell selects the latest valid checkpoint. Keep `START_FROM_SCRATCH = False` to resume an existing checkpoint; use `True` only to start a new run.


In [8]:
import inspect
import math
from collections import Counter
from pathlib import Path

import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
    set_seed,
)

PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the project folder containing data/processed.")

DATA_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "TR-News_6000_summary_length_content_balanced.csv.gz"
)
RUN_TRAINING = False
RUN_PREDICTION_GENERATION = False

MODEL_NAME = "Qwen/Qwen3-0.6B"
OUTPUT_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "qwen3-0.6b-tr-news-lora-6000-summary-length-balanced"
)
FINAL_ADAPTER_DIR = OUTPUT_DIR / "final"

SEED = 42
MAX_SEQUENCE_LENGTH = 1_024
TARGET_TOTAL_RECORDS = 6_000
SUMMARY_CLASSES = ("short", "medium", "long")
TARGET_CLASS_COUNTS = {"short": 1_200, "medium": 3_600, "long": 1_200}
CONTENT_BIN_WIDTH = 25
CONTENT_BIN_COUNT = 12
CONTENT_BIN_LABELS = tuple(
    f"{index * CONTENT_BIN_WIDTH + 1}-{(index + 1) * CONTENT_BIN_WIDTH}"
    for index in range(CONTENT_BIN_COUNT)
)
VALIDATION_FRACTION = 0.0

NUM_EPOCHS = 1
BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 2e-4
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "v_proj"]

if RUN_TRAINING:
    if not torch.cuda.is_available():
        raise RuntimeError(
            "A CUDA-enabled PyTorch environment is required for Qwen3 LoRA training."
        )
    set_seed(SEED)
    print("GPU:", torch.cuda.get_device_name(0))
    print("PyTorch:", torch.__version__)
else:
    print("RUN_TRAINING=False: model loading and training are skipped.")

print("Training data file:", DATA_FILE.name)
print("Adapter output folder:", OUTPUT_DIR.name)

if not DATA_FILE.is_file():
    raise FileNotFoundError(f"Prepared training data not found: {DATA_FILE}")

dataset = load_dataset("csv", data_files=str(DATA_FILE), split="train")
if len(dataset) != TARGET_TOTAL_RECORDS:
    raise ValueError(
        f"Expected {TARGET_TOTAL_RECORDS:,} records, found {len(dataset):,}."
    )

required_columns = {
    "summary_class",
    "instruction",
    "instruction_position",
    "content",
    "abstract",
    "content_words",
    "abstract_words",
    "total_words",
    "content_length_bin",
    "token_count_upper_bound",
}
missing_columns = required_columns.difference(dataset.column_names)
if missing_columns:
    raise ValueError(f"Training data is missing columns: {sorted(missing_columns)}")

class_counts = Counter(dataset["summary_class"])
if dict(class_counts) != TARGET_CLASS_COUNTS:
    raise ValueError(f"Unexpected summary-class counts: {dict(class_counts)}")
position_counts = Counter(dataset["instruction_position"])
if position_counts != Counter({"prefix": 4_800, "suffix": 1_200}):
    raise ValueError(f"Unexpected instruction-position counts: {dict(position_counts)}")
if max(dataset["total_words"]) > 390:
    raise ValueError("Training data contains a record above the 390-word limit.")
if max(dataset["token_count_upper_bound"]) > MAX_SEQUENCE_LENGTH:
    raise ValueError("Training data contains a record above the token limit.")

if VALIDATION_FRACTION > 0:
    validation_size = max(1, round(len(dataset) * VALIDATION_FRACTION))
    split_dataset = dataset.train_test_split(test_size=validation_size, seed=SEED)
    train_raw = split_dataset["train"]
    eval_raw = split_dataset["test"]
else:
    train_raw = dataset
    eval_raw = None

print(f"Train records: {len(train_raw):,}")
print(f"Validation records: {len(eval_raw) if eval_raw is not None else 0:,}")


RUN_TRAINING=False: model loading and training are skipped.
Training data file: TR-News_6000_summary_length_content_balanced.csv.gz
Adapter output folder: qwen3-0.6b-tr-news-lora-6000-summary-length-balanced


Generating train split: 0 examples [00:00, ? examples/s]

Train records: 6,000
Validation records: 0


### Tokenization and loss masking

The Qwen3 chat template is used with thinking disabled. The instruction precedes the article in 80% of records and follows it in 20%. Examples that exceed 1,024 combined prompt-and-target tokens are excluded; inputs are not truncated. The preparation step checks both instruction positions using the longest instruction variant in each class. Prompt tokens are masked from the loss, so the model is trained to predict the abstract.


In [9]:
from collections import Counter

if RUN_TRAINING:

    # Reuse the tokenizer loaded during data preparation when possible.
    tokenizer = globals().get("data_tokenizer")
    if tokenizer is None or getattr(tokenizer, "name_or_path", None) != MODEL_NAME:
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    IM_END_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")
    if IM_END_ID is None or IM_END_ID == tokenizer.unk_token_id:
        raise ValueError("The tokenizer does not define the Qwen <|im_end|> token.")


    def to_token_id_list(token_output):
        """Normalize a single-example tokenizer output to a list of token IDs."""
        if hasattr(token_output, "get"):
            token_output = token_output.get("input_ids", token_output)
        if hasattr(token_output, "tolist"):
            token_output = token_output.tolist()
        if isinstance(token_output, tuple):
            token_output = list(token_output)
        if token_output and isinstance(token_output[0], (list, tuple)):
            if len(token_output) != 1:
                raise ValueError("Expected token IDs for exactly one example.")
            token_output = token_output[0]
        return [int(token_id) for token_id in token_output]


    def tokenize_example(example):
        """Build the chat-formatted example and mask the prompt from the loss."""
        instruction = example["instruction"]
        content = example["content"]
        instruction_position = example["instruction_position"]

        if instruction_position == "prefix":
            user_content = f"{instruction}\n\n{content}"
        elif instruction_position == "suffix":
            user_content = f"{content}\n\n{instruction}"
        else:
            raise ValueError(f"Unexpected instruction position: {instruction_position!r}")

        user_message = {"role": "user", "content": user_content}
        prompt_output = tokenizer.apply_chat_template(
            [user_message],
            tokenize=True,
            add_generation_prompt=True,
            enable_thinking=False,
            return_dict=False,
        )
        prompt_ids = to_token_id_list(prompt_output)

        response_output = tokenizer(example["abstract"], add_special_tokens=False)
        response_ids = to_token_id_list(response_output) + [int(IM_END_ID)]

        full_ids = prompt_ids + response_ids
        total_tokens = len(full_ids)
        labels = [-100] * len(prompt_ids) + response_ids

        return {
            "input_ids": full_ids,
            "attention_mask": [1] * total_tokens,
            "labels": labels,
            "drop_for_length": total_tokens > MAX_SEQUENCE_LENGTH,
            "total_tokens": total_tokens,
            "summary_class": example["summary_class"],
            "content_length_bin": example["content_length_bin"],
        }


    train_tokens = train_raw.map(
        tokenize_example,
        remove_columns=train_raw.column_names,
        desc="Tokenizing training records",
    )
    train_over_limit = sum(train_tokens["drop_for_length"])
    train_tokens = train_tokens.filter(lambda example: not example["drop_for_length"])

    train_class_bin_counts = Counter(
        zip(train_tokens["summary_class"], train_tokens["content_length_bin"])
    )
    print("\nTraining distribution after the 1,024-token filter:")
    print(f"{'Content words':<14}{'Short':>10}{'Medium':>10}{'Long':>10}{'Total':>10}")
    for content_bin in CONTENT_BIN_LABELS:
        counts = [
            train_class_bin_counts[(summary_class, content_bin)]
            for summary_class in SUMMARY_CLASSES
        ]
        print(
            f"{content_bin:<14}{counts[0]:>10,}{counts[1]:>10,}"
            f"{counts[2]:>10,}{sum(counts):>10,}"
        )

    train_tokens = train_tokens.remove_columns(
        ["drop_for_length", "total_tokens", "summary_class", "content_length_bin"]
    )

    if eval_raw is not None:
        eval_tokens = eval_raw.map(
            tokenize_example,
            remove_columns=eval_raw.column_names,
            desc="Tokenizing validation records",
        )
        eval_over_limit = sum(eval_tokens["drop_for_length"])
        eval_tokens = eval_tokens.filter(lambda example: not example["drop_for_length"])
        eval_tokens = eval_tokens.remove_columns(
            ["drop_for_length", "total_tokens", "summary_class", "content_length_bin"]
        )
    else:
        eval_tokens = None
        eval_over_limit = 0

    print(f"Training records excluded above {MAX_SEQUENCE_LENGTH} tokens: {train_over_limit:,}")
    print(f"Training records retained: {len(train_tokens):,}")
    print(f"Validation records excluded above {MAX_SEQUENCE_LENGTH} tokens: {eval_over_limit:,}")
else:
    print("Training disabled: tokenization and loss-mask preparation are skipped.")


Training disabled: tokenization and loss-mask preparation are skipped.


### LoRA model and Trainer settings

LoRA uses `r=16`, `alpha=32`, `dropout=0.05`, and the `q_proj` and `v_proj` target modules. BF16 is used when `torch.cuda.is_bf16_supported()` returns `True`; otherwise, training uses FP16. The micro-batch size is 1 with gradient accumulation over 8 steps (effective batch size 8). Training runs for one epoch with a learning rate of `2e-4`, the AdamW optimizer, a cosine learning-rate schedule, 3% warmup, and weight decay of `0.01`. Hourly and epoch-end checkpoints are saved, with at most two checkpoints retained.

The recorded run completed 750 optimizer steps and reported a training loss of 1.676.


In [10]:
if RUN_TRAINING:
    import time
    from transformers import TrainerCallback

    use_bf16 = torch.cuda.is_bf16_supported()
    compute_dtype = torch.bfloat16 if use_bf16 else torch.float16

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        dtype=compute_dtype,
        low_cpu_mem_usage=True,
    )
    model.config.use_cache = False
    model.config.pad_token_id = tokenizer.pad_token_id
    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )
    model.enable_input_require_grads()

    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=LORA_TARGET_MODULES,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()


    class HourlyCheckpointCallback(TrainerCallback):
        def __init__(self, interval_seconds=3600):
            self.interval_seconds = interval_seconds
            self.last_save_time = None

        def on_train_begin(self, args, state, control, **kwargs):
            self.last_save_time = time.monotonic()
            return control

        def on_step_end(self, args, state, control, **kwargs):
            now = time.monotonic()
            if now - self.last_save_time >= self.interval_seconds:
                control.should_save = True
            return control

        def on_save(self, args, state, control, **kwargs):
            self.last_save_time = time.monotonic()
            print(f"Checkpoint saved at global step {state.global_step:,}.")
            return control


    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        label_pad_token_id=-100,
        pad_to_multiple_of=8,
    )

    training_parameters = inspect.signature(TrainingArguments.__init__).parameters

    total_update_steps = math.ceil(
        len(train_tokens) / (BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)
    )

    training_kwargs = {
        "output_dir": str(OUTPUT_DIR),
        "num_train_epochs": NUM_EPOCHS,
        "per_device_train_batch_size": BATCH_SIZE,
        "per_device_eval_batch_size": BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "learning_rate": LEARNING_RATE,
        "lr_scheduler_type": "cosine",
        "weight_decay": 0.01,
        "max_grad_norm": 1.0,
        "fp16": not use_bf16,
        "bf16": use_bf16,
        "gradient_checkpointing": True,
        "optim": "adamw_torch",
        "logging_steps": 10,
        "save_strategy": "epoch",
        "save_total_limit": 2,
        "load_best_model_at_end": False,
        "group_by_length": True,
        "report_to": "none",
        "dataloader_num_workers": 0,
        "remove_unused_columns": False,
        "seed": SEED,
    }

    if "warmup_ratio" in training_parameters:
        training_kwargs["warmup_ratio"] = 0.03
    elif "warmup_steps" in training_parameters:
        training_kwargs["warmup_steps"] = math.ceil(total_update_steps * 0.03)
    else:
        print("Warning: this Transformers version supports neither warmup_ratio nor warmup_steps.")

    if "eval_strategy" in training_parameters:
        training_kwargs["eval_strategy"] = "no" if eval_tokens is None else "epoch"
    elif "evaluation_strategy" in training_parameters:
        training_kwargs["evaluation_strategy"] = "no" if eval_tokens is None else "epoch"
    elif eval_tokens is not None:
        print("Warning: this Transformers version does not support an evaluation strategy argument.")

    if "gradient_checkpointing_kwargs" in training_parameters:
        training_kwargs["gradient_checkpointing_kwargs"] = {"use_reentrant": False}

    unsupported_training_options = sorted(
        key for key in training_kwargs if key not in training_parameters
    )
    for key in unsupported_training_options:
        training_kwargs.pop(key)
    if unsupported_training_options:
        print(
            "Skipping TrainingArguments options unsupported by this Transformers version:",
            ", ".join(unsupported_training_options),
        )

    training_args = TrainingArguments(**training_kwargs)
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tokens,
        eval_dataset=eval_tokens,
        data_collator=data_collator,
        callbacks=[HourlyCheckpointCallback(interval_seconds=3600)],
    )

    print("Model:", MODEL_NAME)
    print("Sequence length:", MAX_SEQUENCE_LENGTH)
    print("Compute dtype:", compute_dtype)
    print("Micro-batch size:", BATCH_SIZE)
    print("Effective batch size:", BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)
    print("Epochs:", NUM_EPOCHS)
else:
    print("Training disabled: LoRA model and Trainer setup are skipped.")


Training disabled: LoRA model and Trainer setup are skipped.


### Resume or start training and save the adapter

This cell resumes from the latest valid checkpoint, skips training when a completed final adapter is already present, and refuses to start from scratch by default. It is skipped during Run All. To resume an interrupted run intentionally, set `RUN_TRAINING = True`, then run **Data and settings**, **Tokenization and loss mask**, and **LoRA model and Trainer settings** in order before running this cell. For a new training run, also set `START_FROM_SCRATCH = True` in the code cell below.

This cell is skipped by default because `RUN_TRAINING = False`. To resume training intentionally, enable `RUN_TRAINING` in the settings cell first.


In [11]:
if RUN_TRAINING:
    import json

    START_FROM_SCRATCH = False


    def adapter_weights_exist(directory):
        return any(
            (directory / filename).is_file()
            for filename in ("adapter_model.safetensors", "adapter_model.bin")
        )


    checkpoint_candidates = []
    for candidate in OUTPUT_DIR.glob("checkpoint-*"):
        if not candidate.is_dir():
            continue
        try:
            checkpoint_step = int(candidate.name.rsplit("-", 1)[1])
        except (IndexError, ValueError):
            continue
        if not (candidate / "trainer_state.json").is_file():
            continue
        if not adapter_weights_exist(candidate):
            continue
        checkpoint_candidates.append((checkpoint_step, candidate))

    FINAL_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
    training_result = None

    if checkpoint_candidates:
        checkpoint_step, resume_from_checkpoint = max(
            checkpoint_candidates,
            key=lambda item: item[0],
        )

        with (resume_from_checkpoint / "trainer_state.json").open(
            "r",
            encoding="utf-8",
        ) as checkpoint_file:
            checkpoint_state = json.load(checkpoint_file)

        saved_step = int(checkpoint_state.get("global_step", -1))
        saved_max_steps = int(checkpoint_state.get("max_steps", -1))

        if saved_step != checkpoint_step:
            raise ValueError(
                f"Checkpoint folder step {checkpoint_step} does not match "
                f"trainer_state step {saved_step}."
            )
        if saved_max_steps != total_update_steps:
            raise ValueError(
                f"Checkpoint expects {saved_max_steps} total steps, but the current "
                f"training data/settings produce {total_update_steps}."
            )

        if saved_step >= saved_max_steps:
            if adapter_weights_exist(FINAL_ADAPTER_DIR):
                print(
                    f"Training is already complete ({saved_step}/{saved_max_steps}); "
                    "using the existing final adapter."
                )
            else:
                raise RuntimeError(
                    "The checkpoint is complete, but the final adapter is missing. "
                    "Restore the final adapter or select an earlier checkpoint."
                )
        else:
            print(f"Resuming from checkpoint: {resume_from_checkpoint}")
            print(f"Saved progress: {saved_step}/{saved_max_steps} steps")
            training_result = trainer.train(
                resume_from_checkpoint=str(resume_from_checkpoint)
            )
    elif adapter_weights_exist(FINAL_ADAPTER_DIR) and not START_FROM_SCRATCH:
        print("No checkpoint was found; using the existing final adapter.")
    elif START_FROM_SCRATCH:
        print("No checkpoint was found. Starting a new training run as requested.")
        training_result = trainer.train()
    else:
        raise FileNotFoundError(
            f"No valid checkpoint or final adapter was found in {OUTPUT_DIR}. "
            "Set START_FROM_SCRATCH = True to begin a new training run."
        )

    if training_result is not None:
        trainer.save_model(str(FINAL_ADAPTER_DIR))
        tokenizer.save_pretrained(FINAL_ADAPTER_DIR)
        trainer.save_state()
        print("Training run complete.")
        print("Adapter saved to:", FINAL_ADAPTER_DIR)
        print("Training metrics:", training_result.metrics)
    else:
        print("Final adapter:", FINAL_ADAPTER_DIR)
else:
    print("Training disabled: checkpoint inspection and trainer.train() are skipped.")


Training disabled: checkpoint inspection and trainer.train() are skipped.


## 5. Build a fixed evaluation set

A fixed set of 120 held-out articles is selected from the validation split: ten examples from each 25-word content-length bin between 1 and 300 words. The same examples are reused for every instruction and both models. Each selected prompt plus the 192-token generation allowance fits within the 1,024-token evaluation limit.


In [12]:
import csv
import gzip
import random
import sys
from pathlib import Path

from transformers import AutoTokenizer


BASE_MODEL_NAME = "Qwen/Qwen3-0.6B"
RANDOM_SEED = 42
SAMPLES_PER_CONTENT_BIN = 10
MAX_SEQUENCE_LENGTH = 1024
MAX_NEW_TOKENS = 192

EVALUATION_PROMPTS = {
    "short": "Kısa özetle.",
    "medium": "Özetle.",
    "long": "Uzun özetle.",
}

PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "raw").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project folder containing data/raw.")

VALIDATION_FILE = (
    PROJECT_ROOT / "data" / "raw" / "TR-News_validation_content_abstract.csv.gz"
)
EVALUATION_FILE = (
    PROJECT_ROOT / "data" / "processed" / "TR-News_evaluation_120.csv"
)

if not VALIDATION_FILE.is_file():
    raise FileNotFoundError(f"Validation data file not found: {VALIDATION_FILE}")

# Reuse the Qwen tokenizer from the training notebook when available.
eval_tokenizer = globals().get("tokenizer")
tokenizer_name = str(getattr(eval_tokenizer, "name_or_path", "")).lower()

if eval_tokenizer is None or "qwen3-0.6b" not in tokenizer_name:
    eval_tokenizer = AutoTokenizer.from_pretrained(
        BASE_MODEL_NAME,
        use_fast=True,
    )


def get_prompt_token_ids(instruction, content):
    """Tokenize one instruction and article with the model's chat template."""
    messages = [
        {
            "role": "user",
            "content": f"{instruction}\n\n{content}",
        }
    ]

    token_ids = eval_tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_dict=False,
    )

    if hasattr(token_ids, "tolist"):
        token_ids = token_ids.tolist()

    if token_ids and isinstance(token_ids[0], (list, tuple)):
        token_ids = token_ids[0]

    return [int(token_id) for token_id in token_ids]


content_bins = [
    (f"{lower}-{lower + 24}", lower, lower + 24)
    for lower in range(1, 301, 25)
]
records_by_bin = {label: [] for label, _, _ in content_bins}

csv.field_size_limit(2**31 - 1)

with gzip.open(VALIDATION_FILE, "rt", encoding="utf-8", newline="") as source:
    reader = csv.DictReader(source)

    if reader.fieldnames != ["content", "abstract"]:
        raise ValueError(f"Unexpected validation columns: {reader.fieldnames}")

    for row_number, row in enumerate(reader, start=1):
        content = (row["content"] or "").strip()
        abstract = (row["abstract"] or "").strip()

        content_words = len(content.split())
        abstract_words = len(abstract.split())

        if not content or not abstract:
            continue

        if not (1 <= content_words <= 300):
            continue

        if content_words + abstract_words > 390:
            continue

        prompt_lengths = [
            len(get_prompt_token_ids(instruction, content))
            for instruction in EVALUATION_PROMPTS.values()
        ]

        if max(prompt_lengths) + MAX_NEW_TOKENS > MAX_SEQUENCE_LENGTH:
            continue

        bin_index = (content_words - 1) // 25
        bin_label = content_bins[bin_index][0]

        records_by_bin[bin_label].append(
            {
                "validation_row": row_number,
                "content_bin": bin_label,
                "content_words": content_words,
                "abstract_words": abstract_words,
                "content": content,
                "abstract": abstract,
            }
        )

available_counts = {
    label: len(records_by_bin[label])
    for label, _, _ in content_bins
}

samples_per_bin = min(
    SAMPLES_PER_CONTENT_BIN,
    min(available_counts.values()),
)

if samples_per_bin == 0:
    raise ValueError(
        "At least one content-length bin has no usable validation examples."
    )

rng = random.Random(RANDOM_SEED)
evaluation_records = []

for label, _, _ in content_bins:
    evaluation_records.extend(
        rng.sample(records_by_bin[label], samples_per_bin)
    )

EVALUATION_FILE.parent.mkdir(parents=True, exist_ok=True)

fieldnames = [
    "validation_row",
    "content_bin",
    "content_words",
    "abstract_words",
    "content",
    "abstract",
]

with EVALUATION_FILE.open("w", encoding="utf-8", newline="") as destination:
    writer = csv.DictWriter(destination, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(evaluation_records)

print(f"Saved evaluation set: {EVALUATION_FILE.name}")
print(f"Examples per content-length bin: {samples_per_bin}")
print(f"Total evaluation examples: {len(evaluation_records)}")
print()
print(f"{'Content words':<18}{'Available':>12}{'Selected':>12}")

for label, _, _ in content_bins:
    print(
        f"{label:<18}"
        f"{available_counts[label]:>12,}"
        f"{samples_per_bin:>12,}"
    )

Saved evaluation set: TR-News_evaluation_120.csv
Examples per content-length bin: 10
Total evaluation examples: 120

Content words        Available    Selected
1-25                       106          10
26-50                      732          10
51-75                    1,656          10
76-100                   1,806          10
101-125                  1,544          10
126-150                  1,194          10
151-175                  1,070          10
176-200                    850          10
201-225                    716          10
226-250                    565          10
251-275                    533          10
276-300                    378          10


## 6. Generate base and LoRA predictions

The LoRA adapter is merged into the base weights for inference. Both the merged model and the original base model use the same evaluation articles, three canonical instructions, deterministic decoding, 192-token output limit, and batch size of eight. Predictions are saved after each batch so an interrupted evaluation can resume.

New prediction generation is disabled by default. Run All displays cached LoRA prediction examples and uses the saved prediction file for metric scoring. Set `RUN_PREDICTION_GENERATION = True` only when you intentionally want to load both models and generate predictions again.


In [ ]:
if RUN_PREDICTION_GENERATION:
    import csv
    import gc
    from pathlib import Path

    import torch
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer


    BASE_MODEL_NAME = "Qwen/Qwen3-0.6B"
    BATCH_SIZE = 8
    MAX_NEW_TOKENS = 192
    MAX_SEQUENCE_LENGTH = 1024

    EVALUATION_PROMPTS = {
        "short": "Kısa özetle.",
        "medium": "Özetle.",
        "long": "Uzun özetle.",
    }

    PROJECT_ROOT = next(
        (
            candidate
            for candidate in (Path.cwd(), *Path.cwd().parents)
            if (candidate / "data" / "processed").is_dir()
        ),
        None,
    )

    if PROJECT_ROOT is None:
        raise FileNotFoundError(
            "Could not find the project folder containing data/processed."
        )

    EVALUATION_FILE = (
        PROJECT_ROOT / "data" / "processed" / "TR-News_evaluation_120.csv"
    )
    ADAPTER_DIR = (
        PROJECT_ROOT
        / "outputs"
        / "qwen3-0.6b-tr-news-lora-6000-summary-length-balanced"
        / "final"
    )
    PREDICTIONS_FILE = (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "TR-News_eval_predictions_120_batch8.csv"
    )

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is required for this evaluation.")

    if not EVALUATION_FILE.is_file():
        raise FileNotFoundError(f"Evaluation set not found: {EVALUATION_FILE}")

    if not ADAPTER_DIR.is_dir():
        raise FileNotFoundError(
            f"Final adapter directory not found: {ADAPTER_DIR}. "
            "Complete training and save the final adapter before evaluation."
        )

    if not (ADAPTER_DIR / "adapter_config.json").is_file():
        raise FileNotFoundError(f"Adapter configuration not found in {ADAPTER_DIR}")

    if not any(
        (ADAPTER_DIR / filename).is_file()
        for filename in ("adapter_model.safetensors", "adapter_model.bin")
    ):
        raise FileNotFoundError(f"Adapter weights not found in {ADAPTER_DIR}")


    evaluation_records = []

    with EVALUATION_FILE.open("r", encoding="utf-8", newline="") as source:
        reader = csv.DictReader(source)

        required_columns = {
            "validation_row",
            "content_bin",
            "content_words",
            "abstract_words",
            "content",
            "abstract",
        }
        missing_columns = required_columns.difference(reader.fieldnames or [])

        if missing_columns:
            raise ValueError(
                f"Evaluation file is missing columns: {sorted(missing_columns)}"
            )

        for row in reader:
            evaluation_records.append(
                {
                    "validation_row": str(row["validation_row"]),
                    "content_bin": row["content_bin"],
                    "content_words": int(row["content_words"]),
                    "abstract_words": int(row["abstract_words"]),
                    "content": row["content"],
                    "abstract": row["abstract"],
                }
            )

    if len(evaluation_records) != 120:
        raise ValueError(
            f"Expected 120 evaluation examples, found {len(evaluation_records)}."
        )

    # Preserve the precision selection used in the training notebook.
    use_bf16 = torch.cuda.is_bf16_supported()
    compute_dtype = torch.bfloat16 if use_bf16 else torch.float16

    eval_tokenizer = AutoTokenizer.from_pretrained(
        BASE_MODEL_NAME,
        use_fast=True,
    )

    if eval_tokenizer.pad_token is None:
        eval_tokenizer.pad_token = eval_tokenizer.eos_token

    if eval_tokenizer.pad_token_id is None:
        raise ValueError("The tokenizer does not define a padding token.")

    # Decoder-only models use left padding for batched generation.
    eval_tokenizer.padding_side = "left"

    im_end_id = eval_tokenizer.convert_tokens_to_ids("<|im_end|>")
    if im_end_id is None or im_end_id == eval_tokenizer.unk_token_id:
        raise ValueError("The tokenizer does not define the Qwen <|im_end|> token.")


    PREDICTION_COLUMNS = [
        "model_variant",
        "instruction_type",
        "instruction",
        "validation_row",
        "content_bin",
        "content_words",
        "abstract_words",
        "generated_tokens",
        "prediction_words",
        "content",
        "reference",
        "prediction",
    ]

    completed_keys = set()

    if PREDICTIONS_FILE.is_file() and PREDICTIONS_FILE.stat().st_size > 0:
        with PREDICTIONS_FILE.open("r", encoding="utf-8", newline="") as source:
            reader = csv.DictReader(source)

            if reader.fieldnames != PREDICTION_COLUMNS:
                raise ValueError(
                    f"Existing predictions file has an unexpected schema: "
                    f"{PREDICTIONS_FILE}"
                )

            for row in reader:
                completed_keys.add(
                    (
                        row["model_variant"],
                        row["instruction_type"],
                        row["validation_row"],
                    )
                )


    def build_prompt_ids(instruction, content):
        """Create token IDs using the same Qwen chat template as training."""
        messages = [
            {
                "role": "user",
                "content": f"{instruction}\n\n{content}",
            }
        ]

        prompt_ids = eval_tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            enable_thinking=False,
            return_dict=False,
        )

        if hasattr(prompt_ids, "tolist"):
            prompt_ids = prompt_ids.tolist()

        if prompt_ids and isinstance(prompt_ids[0], (list, tuple)):
            prompt_ids = prompt_ids[0]

        return [int(token_id) for token_id in prompt_ids]


    def generate_batch(model, instruction, batch_records):
        """Generate summaries for one batch and return predictions with token counts."""
        prompt_sequences = [
            build_prompt_ids(instruction, record["content"])
            for record in batch_records
        ]

        longest_prompt = max(len(sequence) for sequence in prompt_sequences)

        if longest_prompt + MAX_NEW_TOKENS > MAX_SEQUENCE_LENGTH:
            raise ValueError(
                "A prompt in this batch exceeds the configured 1,024-token limit."
            )

        input_device = model.get_input_embeddings().weight.device
        pad_token_id = eval_tokenizer.pad_token_id

        input_ids = torch.full(
            (len(prompt_sequences), longest_prompt),
            pad_token_id,
            dtype=torch.long,
            device=input_device,
        )
        attention_mask = torch.zeros_like(input_ids)

        for row_index, token_ids in enumerate(prompt_sequences):
            token_tensor = torch.tensor(
                token_ids,
                dtype=torch.long,
                device=input_device,
            )
            input_ids[row_index, -len(token_ids):] = token_tensor
            attention_mask[row_index, -len(token_ids):] = 1

        with torch.inference_mode():
            output_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                num_beams=1,
                eos_token_id=im_end_id,
                pad_token_id=pad_token_id,
                use_cache=True,
            )

        predictions = []

        for row_index in range(len(batch_records)):
            generated_ids = output_ids[row_index, longest_prompt:]
            generated_id_list = generated_ids.tolist()

            if im_end_id in generated_id_list:
                stop_index = generated_id_list.index(im_end_id)
                counted_ids = generated_id_list[: stop_index + 1]
            else:
                counted_ids = generated_id_list

            prediction = eval_tokenizer.decode(
                generated_ids,
                skip_special_tokens=True,
            ).strip()

            predictions.append((prediction, len(counted_ids)))

        return predictions


    def append_prediction_rows(rows):
        """Append a completed batch so interrupted runs can resume safely."""
        file_exists = (
            PREDICTIONS_FILE.is_file()
            and PREDICTIONS_FILE.stat().st_size > 0
        )

        with PREDICTIONS_FILE.open("a", encoding="utf-8", newline="") as destination:
            writer = csv.DictWriter(
                destination,
                fieldnames=PREDICTION_COLUMNS,
            )

            if not file_exists:
                writer.writeheader()

            writer.writerows(rows)
            destination.flush()


    def evaluate_model(model, model_variant):
        """Evaluate one model using all three prompts and the fixed evaluation set."""
        model.eval()
        model.config.use_cache = True
        model.config.pad_token_id = eval_tokenizer.pad_token_id

        for instruction_type, instruction in EVALUATION_PROMPTS.items():
            pending_records = [
                record
                for record in evaluation_records
                if (
                    model_variant,
                    instruction_type,
                    record["validation_row"],
                )
                not in completed_keys
            ]

            if not pending_records:
                print(f"Already complete: {model_variant} / {instruction_type}")
                continue

            batch_count = (
                len(pending_records) + BATCH_SIZE - 1
            ) // BATCH_SIZE

            print(
                f"Starting {model_variant} / {instruction_type}: "
                f"{len(pending_records)} examples in {batch_count} batches."
            )

            for batch_number, start_index in enumerate(
                range(0, len(pending_records), BATCH_SIZE),
                start=1,
            ):
                batch_records = pending_records[
                    start_index : start_index + BATCH_SIZE
                ]
                batch_predictions = generate_batch(
                    model,
                    instruction,
                    batch_records,
                )

                batch_rows = []

                for record, (prediction, generated_tokens) in zip(
                    batch_records,
                    batch_predictions,
                ):
                    batch_rows.append(
                        {
                            "model_variant": model_variant,
                            "instruction_type": instruction_type,
                            "instruction": instruction,
                            "validation_row": record["validation_row"],
                            "content_bin": record["content_bin"],
                            "content_words": record["content_words"],
                            "abstract_words": record["abstract_words"],
                            "generated_tokens": generated_tokens,
                            "prediction_words": len(prediction.split()),
                            "content": record["content"],
                            "reference": record["abstract"],
                            "prediction": prediction,
                        }
                    )

                append_prediction_rows(batch_rows)

                for row in batch_rows:
                    completed_keys.add(
                        (
                            row["model_variant"],
                            row["instruction_type"],
                            row["validation_row"],
                        )
                    )

                print(
                    f"{model_variant} / {instruction_type}: "
                    f"batch {batch_number}/{batch_count} saved."
                )


    print("Loading base model and LoRA adapter for merging.")
    base_for_merge = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        dtype=compute_dtype,
        device_map="auto",
        low_cpu_mem_usage=True,
    )

    peft_model = PeftModel.from_pretrained(
        base_for_merge,
        str(ADAPTER_DIR),
        is_trainable=False,
    )

    merged_model = peft_model.merge_and_unload(safe_merge=True)
    merged_model.eval()
    merged_model.config.use_cache = True

    del peft_model
    del base_for_merge
    gc.collect()
    torch.cuda.empty_cache()

    print("Evaluating the merged LoRA model with batch size 8.")
    evaluate_model(merged_model, "lora_merged")

    del merged_model
    gc.collect()
    torch.cuda.empty_cache()

    print("Loading the original base model for comparison.")
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        dtype=compute_dtype,
        device_map="auto",
        low_cpu_mem_usage=True,
    )

    base_model.eval()
    base_model.config.use_cache = True

    print("Evaluating the base model with batch size 8.")
    evaluate_model(base_model, "base")

    del base_model
    gc.collect()
    torch.cuda.empty_cache()

    expected_predictions = (
        len(evaluation_records) * len(EVALUATION_PROMPTS) * 2
    )

    print(f"Saved predictions: {PREDICTIONS_FILE}")
    print(f"Completed: {len(completed_keys):,} / {expected_predictions:,}")
else:
    import pandas as pd
    from pathlib import Path

    prediction_project_root = next(
        (
            candidate
            for candidate in (Path.cwd(), *Path.cwd().parents)
            if (candidate / "data" / "processed").is_dir()
        ),
        None,
    )
    if prediction_project_root is None:
        print("Prediction generation is disabled; no processed-data folder was found.")
    else:
        cached_predictions_file = (
            prediction_project_root
            / "data"
            / "processed"
            / "TR-News_eval_predictions_120_batch8.csv"
        )
        if not cached_predictions_file.is_file():
            print(
                "Prediction generation is disabled and no saved prediction file was found. "
                "Set RUN_PREDICTION_GENERATION = True to generate predictions."
            )
        else:
            cached_predictions = pd.read_csv(cached_predictions_file)
            lora_predictions = cached_predictions[
                cached_predictions["model_variant"] == "lora_merged"
            ]
            print(
                "Prediction generation is disabled. Showing saved LoRA examples from "
                f"{cached_predictions_file.name}."
            )
            for instruction_type in ("short", "medium", "long"):
                example_rows = lora_predictions[
                    lora_predictions["instruction_type"] == instruction_type
                ]
                if example_rows.empty:
                    continue
                example = example_rows.iloc[0]
                content_preview = str(example["content"]).replace("\n", " ").strip()
                if len(content_preview) > 450:
                    content_preview = content_preview[:447].rstrip() + "..."
                print(f"\n--- {instruction_type.title()} instruction ---")
                print("Instruction:", example["instruction"])
                print("Article excerpt:", content_preview)
                print("Reference summary:", example["reference"])
                print("LoRA prediction:", example["prediction"])


## 7. Score the generated summaries

The dependency-check cell verifies that BERTScore and ROUGE are available without installing packages during Run All. BERTScore uses the Turkish BERTurk checkpoint; ROUGE-1, ROUGE-2, and ROUGE-L are computed against the dataset abstracts.


In [14]:
try:
    import bert_score
    import rouge_score
    print("BERTScore and ROUGE dependencies are available.")
except ImportError as error:
    raise ImportError(
        "Install evaluation dependencies once with: `%pip install bert-score rouge-score`"
    ) from error


BERTScore and ROUGE dependencies are available.


In [15]:
import gc
from pathlib import Path

import pandas as pd
import torch
from bert_score import score as bert_score
from rouge_score import rouge_scorer


PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find the project folder containing data/processed."
    )

PREDICTIONS_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "TR-News_eval_predictions_120_batch8.csv"
)
SCORED_PREDICTIONS_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "TR-News_eval_predictions_120_batch8_scored.csv"
)
METRICS_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "TR-News_eval_metrics_120_batch8.csv"
)
LENGTH_CONTROL_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "TR-News_eval_length_control_120_batch8.csv"
)

BERTSCORE_MODEL = "dbmdz/bert-base-turkish-cased"
BERTSCORE_BATCH_SIZE = 8
EXPECTED_ROWS = 720

if not PREDICTIONS_FILE.is_file():
    raise FileNotFoundError(f"Prediction file not found: {PREDICTIONS_FILE}")

predictions_df = pd.read_csv(PREDICTIONS_FILE)

if len(predictions_df) != EXPECTED_ROWS:
    raise ValueError(
        f"Expected {EXPECTED_ROWS} prediction rows, found {len(predictions_df)}."
    )

required_columns = {
    "model_variant",
    "instruction_type",
    "validation_row",
    "content_words",
    "abstract_words",
    "prediction_words",
    "reference",
    "prediction",
}
missing_columns = required_columns.difference(predictions_df.columns)

if missing_columns:
    raise ValueError(
        f"Prediction file is missing columns: {sorted(missing_columns)}"
    )

duplicate_rows = predictions_df.duplicated(
    subset=["model_variant", "instruction_type", "validation_row"]
)

if duplicate_rows.any():
    raise ValueError(
        f"Found {int(duplicate_rows.sum())} duplicate prediction rows."
    )

predictions_df["prediction"] = predictions_df["prediction"].fillna("").astype(str)
predictions_df["reference"] = predictions_df["reference"].fillna("").astype(str)

empty_predictions = predictions_df["prediction"].str.strip().eq("")
if empty_predictions.any():
    raise ValueError(
        f"Found {int(empty_predictions.sum())} empty generated summaries. "
        "Inspect those rows before scoring."
    )

# Release unused GPU cache before loading the BERTScore model.
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

candidates = predictions_df["prediction"].tolist()
references = predictions_df["reference"].tolist()

bert_precision, bert_recall, bert_f1 = bert_score(
    candidates,
    references,
    model_type=BERTSCORE_MODEL,
    lang="tr",
    device="cuda" if torch.cuda.is_available() else "cpu",
    batch_size=BERTSCORE_BATCH_SIZE,
    verbose=True,
    rescale_with_baseline=False,
)

predictions_df["bertscore_precision"] = bert_precision.detach().cpu().tolist()
predictions_df["bertscore_recall"] = bert_recall.detach().cpu().tolist()
predictions_df["bertscore_f1"] = bert_f1.detach().cpu().tolist()

rouge = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=False,
)

rouge_results = []

for reference, prediction in zip(references, candidates):
    scores = rouge.score(reference, prediction)
    rouge_results.append(
        {
            "rouge1_f1": scores["rouge1"].fmeasure,
            "rouge2_f1": scores["rouge2"].fmeasure,
            "rougeL_f1": scores["rougeL"].fmeasure,
        }
    )

rouge_df = pd.DataFrame(rouge_results)
predictions_df = pd.concat(
    [predictions_df.reset_index(drop=True), rouge_df],
    axis=1,
)

predictions_df["prediction_words"] = predictions_df["prediction"].map(
    lambda text: len(text.split())
)

predictions_df["prediction_content_ratio_pct"] = (
    predictions_df["prediction_words"]
    / predictions_df["content_words"]
    * 100
)

group_columns = ["model_variant", "instruction_type"]
metric_columns = [
    "bertscore_precision",
    "bertscore_recall",
    "bertscore_f1",
    "rouge1_f1",
    "rouge2_f1",
    "rougeL_f1",
]

metrics_df = (
    predictions_df
    .groupby(group_columns, as_index=False)
    .agg(
        example_count=("validation_row", "count"),
        **{
            f"mean_{column}": (column, "mean")
            for column in metric_columns
        },
        mean_prediction_words=("prediction_words", "mean"),
        median_prediction_words=("prediction_words", "median"),
        mean_reference_words=("abstract_words", "mean"),
        mean_prediction_content_ratio_pct=(
            "prediction_content_ratio_pct",
            "mean",
        ),
        total_prediction_words=("prediction_words", "sum"),
        total_content_words=("content_words", "sum"),
    )
)

metrics_df["overall_prediction_content_ratio_pct"] = (
    100
    * metrics_df["total_prediction_words"]
    / metrics_df["total_content_words"]
)

instruction_order = ["short", "medium", "long"]
metrics_df["instruction_type"] = pd.Categorical(
    metrics_df["instruction_type"],
    categories=instruction_order,
    ordered=True,
)
metrics_df = metrics_df.sort_values(
    ["model_variant", "instruction_type"]
).reset_index(drop=True)

length_pivot = predictions_df.pivot(
    index=["model_variant", "validation_row"],
    columns="instruction_type",
    values="prediction_words",
)

length_control_rows = []

for model_variant in sorted(predictions_df["model_variant"].unique()):
    model_lengths = length_pivot.loc[model_variant].dropna(
        subset=["short", "medium", "long"]
    )

    strict_order = (
        (model_lengths["short"] < model_lengths["medium"])
        & (model_lengths["medium"] < model_lengths["long"])
    )

    non_decreasing_order = (
        (model_lengths["short"] <= model_lengths["medium"])
        & (model_lengths["medium"] <= model_lengths["long"])
    )

    length_control_rows.append(
        {
            "model_variant": model_variant,
            "example_count": len(model_lengths),
            "strict_short_medium_long_rate_pct": (
                strict_order.mean() * 100
            ),
            "non_decreasing_short_medium_long_rate_pct": (
                non_decreasing_order.mean() * 100
            ),
            "mean_short_words": model_lengths["short"].mean(),
            "mean_medium_words": model_lengths["medium"].mean(),
            "mean_long_words": model_lengths["long"].mean(),
        }
    )

length_control_df = pd.DataFrame(length_control_rows)

predictions_df.to_csv(
    SCORED_PREDICTIONS_FILE,
    index=False,
    encoding="utf-8-sig",
)
metrics_df.to_csv(
    METRICS_FILE,
    index=False,
    encoding="utf-8-sig",
)
length_control_df.to_csv(
    LENGTH_CONTROL_FILE,
    index=False,
    encoding="utf-8-sig",
)

print("Metrics by model and instruction:")
print(metrics_df.round(4).to_string(index=False))

print("\nInstruction length control:")
print(length_control_df.round(2).to_string(index=False))

print(f"\nScored predictions saved to: {SCORED_PREDICTIONS_FILE.name}")
print(f"Metric summary saved to: {METRICS_FILE.name}")
print(f"Length-control summary saved to: {LENGTH_CONTROL_FILE.name}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


  0%|          | 0/97 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/90 [00:00<?, ?it/s]

done in 7.41 seconds, 97.18 sentences/sec
Metrics by model and instruction:
model_variant instruction_type  example_count  mean_bertscore_precision  mean_bertscore_recall  mean_bertscore_f1  mean_rouge1_f1  mean_rouge2_f1  mean_rougeL_f1  mean_prediction_words  median_prediction_words  mean_reference_words  mean_prediction_content_ratio_pct  total_prediction_words  total_content_words  overall_prediction_content_ratio_pct
         base            short            120                    0.5119                 0.6590             0.5727          0.2747          0.1539          0.2205                60.6917                     68.0               20.4667                            57.7238                    7283                18019                               40.4184
         base           medium            120                    0.5017                 0.6609             0.5678          0.2648          0.1494          0.2106                65.2333                     70.0               

In [16]:
from pathlib import Path

import pandas as pd
from IPython.display import display


PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find the project folder containing data/processed."
    )

METRICS_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "TR-News_eval_metrics_120_batch8.csv"
)
LENGTH_CONTROL_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "TR-News_eval_length_control_120_batch8.csv"
)

if not METRICS_FILE.is_file():
    raise FileNotFoundError(f"Metrics file not found: {METRICS_FILE}")

if not LENGTH_CONTROL_FILE.is_file():
    raise FileNotFoundError(
        f"Length-control file not found: {LENGTH_CONTROL_FILE}"
    )

metrics_df = pd.read_csv(METRICS_FILE)
length_control_df = pd.read_csv(LENGTH_CONTROL_FILE)

model_labels = {
    "base": "Base model",
    "lora_merged": "LoRA (merged)",
}
instruction_labels = {
    "short": "Short summary",
    "medium": "Summarize",
    "long": "Long summary",
}

model_order = {"base": 0, "lora_merged": 1}
instruction_order = {"short": 0, "medium": 1, "long": 2}

ordered_metrics = metrics_df.copy()
ordered_metrics["_model_order"] = (
    ordered_metrics["model_variant"].map(model_order)
)
ordered_metrics["_instruction_order"] = (
    ordered_metrics["instruction_type"].map(instruction_order)
)
ordered_metrics = ordered_metrics.sort_values(
    ["_model_order", "_instruction_order"]
)

quality_table = ordered_metrics[
    [
        "model_variant",
        "instruction_type",
        "example_count",
        "mean_bertscore_f1",
        "mean_rouge1_f1",
        "mean_rouge2_f1",
        "mean_rougeL_f1",
    ]
].rename(
    columns={
        "model_variant": "Model",
        "instruction_type": "Instruction",
        "example_count": "Examples",
        "mean_bertscore_f1": "BERTScore F1",
        "mean_rouge1_f1": "ROUGE-1 F1",
        "mean_rouge2_f1": "ROUGE-2 F1",
        "mean_rougeL_f1": "ROUGE-L F1",
    }
)

quality_table["Model"] = quality_table["Model"].map(model_labels)
quality_table["Instruction"] = quality_table["Instruction"].map(instruction_labels)

length_table = ordered_metrics[
    [
        "model_variant",
        "instruction_type",
        "mean_prediction_words",
        "median_prediction_words",
        "mean_reference_words",
        "mean_prediction_content_ratio_pct",
        "overall_prediction_content_ratio_pct",
    ]
].rename(
    columns={
        "model_variant": "Model",
        "instruction_type": "Instruction",
        "mean_prediction_words": "Mean summary words",
        "median_prediction_words": "Median summary words",
        "mean_reference_words": "Mean reference words",
        "mean_prediction_content_ratio_pct": "Mean per-example summary/content ratio (%)",
        "overall_prediction_content_ratio_pct": "Aggregate summary/content ratio (%)",
    }
)

length_table["Model"] = length_table["Model"].map(model_labels)
length_table["Instruction"] = length_table["Instruction"].map(instruction_labels)

control_table = length_control_df[
    [
        "model_variant",
        "example_count",
        "strict_short_medium_long_rate_pct",
        "non_decreasing_short_medium_long_rate_pct",
        "mean_short_words",
        "mean_medium_words",
        "mean_long_words",
    ]
].rename(
    columns={
        "model_variant": "Model",
        "example_count": "Examples",
        "strict_short_medium_long_rate_pct": "Strict short < medium < long (%)",
        "non_decreasing_short_medium_long_rate_pct": "Non-decreasing (%)",
        "mean_short_words": "Mean short words",
        "mean_medium_words": "Mean medium words",
        "mean_long_words": "Mean long words",
    }
)

control_table["Model"] = control_table["Model"].map(model_labels)

table_styles = [
    {
        "selector": "table",
        "props": [
            ("border-collapse", "collapse"),
            ("font-family", "Arial, sans-serif"),
            ("font-size", "13px"),
            ("width", "100%"),
        ],
    },
    {
        "selector": "caption",
        "props": [
            ("caption-side", "top"),
            ("font-size", "16px"),
            ("font-weight", "bold"),
            ("text-align", "left"),
            ("padding", "8px 0"),
            ("color", "#172554"),
        ],
    },
    {
        "selector": "th",
        "props": [
            ("background-color", "#1e3a5f"),
            ("color", "white"),
            ("padding", "9px 12px"),
            ("text-align", "center"),
            ("white-space", "nowrap"),
        ],
    },
    {
        "selector": "td",
        "props": [
            ("padding", "8px 12px"),
            ("border-bottom", "1px solid #e5e7eb"),
            ("text-align", "center"),
        ],
    },
    {
        "selector": "tbody tr:nth-child(even)",
        "props": [("background-color", "#f3f6fa")],
    },
]


quality_style = (
    quality_table.style
    .hide(axis="index")
    .set_caption("Summary Quality by Model and Instruction")
    .set_table_styles(table_styles)
    .format(
        {
            "Examples": "{:,.0f}",
            "BERTScore F1": "{:.3f}",
            "ROUGE-1 F1": "{:.3f}",
            "ROUGE-2 F1": "{:.3f}",
            "ROUGE-L F1": "{:.3f}",
        }
    )
    .background_gradient(
        cmap="YlGn",
        subset=[
            "BERTScore F1",
            "ROUGE-1 F1",
            "ROUGE-2 F1",
            "ROUGE-L F1",
        ],
        axis=0,
        vmin=0,
        vmax=1,
    )
)

length_style = (
    length_table.style
    .hide(axis="index")
    .set_caption("Generated Summary Lengths")
    .set_table_styles(table_styles)
    .format(
        {
            "Mean summary words": "{:.1f}",
            "Median summary words": "{:.1f}",
            "Mean reference words": "{:.1f}",
            "Mean per-example summary/content ratio (%)": "{:.1f}%",
            "Aggregate summary/content ratio (%)": "{:.1f}%",
        }
    )
)

control_style = (
    control_table.style
    .hide(axis="index")
    .set_caption("Instruction Length Control")
    .set_table_styles(table_styles)
    .format(
        {
            "Examples": "{:,.0f}",
            "Strict short < medium < long (%)": "{:.1f}%",
            "Non-decreasing (%)": "{:.1f}%",
            "Mean short words": "{:.1f}",
            "Mean medium words": "{:.1f}",
            "Mean long words": "{:.1f}",
        }
    )
    .background_gradient(
        cmap="YlGn",
        subset=[
            "Strict short < medium < long (%)",
            "Non-decreasing (%)",
        ],
        axis=0,
        vmin=0,
        vmax=100,
    )
)

display(quality_style)
display(length_style)
display(control_style)

Model,Instruction,Examples,BERTScore F1,ROUGE-1 F1,ROUGE-2 F1,ROUGE-L F1
Base model,Short summary,120,0.573,0.275,0.154,0.220
Base model,Summarize,120,0.568,0.265,0.149,0.211
Base model,Long summary,120,0.577,0.272,0.155,0.218
LoRA (merged),Short summary,120,0.617,0.318,0.187,0.284
LoRA (merged),Summarize,120,0.628,0.344,0.210,0.298
LoRA (merged),Long summary,120,0.619,0.340,0.206,0.286


Model,Instruction,Mean summary words,Median summary words,Mean reference words,Mean per-example summary/content ratio (%),Aggregate summary/content ratio (%)
Base model,Short summary,60.7,68.0,20.5,57.7%,40.4%
Base model,Summarize,65.2,70.0,20.5,106.2%,43.4%
Base model,Long summary,64.8,69.0,20.5,139.2%,43.2%
LoRA (merged),Short summary,13.1,11.0,20.5,16.4%,8.7%
LoRA (merged),Summarize,19.6,17.5,20.5,89.5%,13.0%
LoRA (merged),Long summary,32.3,27.5,20.5,186.0%,21.5%


Model,Examples,Strict short < medium < long (%),Non-decreasing (%),Mean short words,Mean medium words,Mean long words
Base model,120,16.7%,36.7%,60.7,65.2,64.8
LoRA (merged),120,49.2%,85.8%,13.1,19.6,32.3


## 8. Results and interpretation

The fixed evaluation set contains 120 articles. Each article is summarized with all three instructions by both models, for 720 generated summaries in total.

### Reference-based quality metrics

| Instruction | Model | BERTScore F1 | ROUGE-1 F1 | ROUGE-2 F1 | ROUGE-L F1 |
|---|---|---:|---:|---:|---:|
| Short summary | Base | 0.573 | 0.275 | 0.154 | 0.220 |
| Short summary | LoRA-merged | 0.617 | 0.318 | 0.187 | 0.284 |
| Summarize | Base | 0.568 | 0.265 | 0.149 | 0.211 |
| Summarize | LoRA-merged | 0.628 | 0.344 | 0.210 | 0.298 |
| Long summary | Base | 0.577 | 0.272 | 0.155 | 0.218 |
| Long summary | LoRA-merged | 0.620 | 0.340 | 0.206 | 0.286 |

### Output length and instruction following

| Instruction | Base mean words | LoRA-merged mean words |
|---|---:|---:|
| Short summary | 60.7 | 13.1 |
| Summarize | 65.2 | 19.6 |
| Long summary | 64.8 | 32.3 |

The LoRA model scores higher than the base model on each reported reference-based metric and produces clearly differentiated summary lengths. Its outputs follow a strict short-to-medium-to-long length order on 49.2% of articles and a non-decreasing order on 85.8%.

The same reference abstract is used for all three instructions. BERTScore and ROUGE therefore measure similarity to that reference; the separate output-length results show how well the model follows the requested length style.
